In [2]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

from rapidfuzz import fuzz, process

In [4]:
data = pd.read_csv('transformed/past_house_results.csv')
data.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,...,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2018,Alabama,AL,False,1,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,...,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555
1,2018,Alabama,AL,False,2,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,...,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938
2,2018,Alabama,AL,False,3,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,...,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692
3,2018,Alabama,AL,False,4,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,...,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48,529497.35,11.550741,88.449259
4,2018,Alabama,AL,False,5,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,...,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38,1769886.91,31.266773,68.733227


In [5]:
data.columns

Index(['year', 'state', 'state_po', 'special', 'district', 'dem', 'rep',
       'totalvotes', 'dem_cand', 'rep_cand', 'dem_inc', 'rep_inc', 'dem_funds',
       'rep_funds', '2party_votes', 'dem_pct_2p', 'rep_pct_2p',
       'dem_tot_funds', 'rep_tot_funds', 'tot_funds', 'dem_funds_2p_pct',
       'rep_funds_2p_pct'],
      dtype='object')

In [6]:
demo22_cols = ['district_name', 'district', 'white_pct', 'black_pct', 'hisp_pct', 'asn_pct',
                           'natam_pct', 'pi_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_asn_pct',
                           'vap_natam_pct', 'vap_pi_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_asn_pct',
                           'cit_natam_pct', 'cit_pi_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_asn_pct',
                           'cvap_natam_pct', 'cvap_pi_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'asn_pop', 'natam_pop', 'pi_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'asn_vap_pop',
                           'natam_vap_pop', 'pi_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'asn_cit_pop',
                           'natam_cit_pop', 'pi_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'asn_cvap_pop',
                           'natam_cvap_pop', 'pi_cvap_pop']

demo_cols = ['district_name', 'district', 'white_pct', 'black_pct', 'hisp_pct', 'asn_pct',
                           'natam_pct', 'other_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_asn_pct',
                           'vap_natam_pct', 'vap_other_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_asn_pct',
                           'cit_natam_pct', 'cit_other_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_asn_pct',
                           'cvap_natam_pct', 'cvap_other_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'asn_pop', 'natam_pop', 'other_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'asn_vap_pop',
                           'natam_vap_pop', 'other_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'asn_cit_pop',
                           'natam_cit_pop', 'other_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'asn_cvap_pop',
                           'natam_cvap_pop', 'other_cvap_pop']

In [7]:
demo_18 = pd.read_csv('data/demo/2018_116_acs_demo.csv')
demo_18 = demo_18.iloc[2:]
demo_18 = demo_18.set_axis(demo_cols, axis=1)
demo_18['district'] = demo_18['district'].str.replace('-AL', '-00')
demo_18.head()

,district_name,district,white_pct,black_pct,hisp_pct,asn_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,...,asn_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,asn_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama's 1st,AL-01,65.2,27.6,3.2,1.5,1.0,1.4,67.6,26.0,...,"7,770","7,255","10,150","534,355","365,070","140,815","11,025","5,710","5,840","5,895"
3,Alabama's 2nd,AL-02,61.9,31.2,3.6,1.2,0.4,1.9,64.3,30.1,...,"4,610","2,540","12,605","516,040","336,825","157,445","9,980","3,250","2,045","6,495"
4,Alabama's 3rd,AL-03,67.8,25.4,3.1,1.8,0.3,1.6,69.5,24.9,...,"6,935","1,975","11,610","538,125","380,720","135,955","9,705","4,490","1,610","5,640"
5,Alabama's 4th,AL-04,83.7,6.9,6.5,0.6,0.7,1.6,85.9,6.9,...,"2,795","4,775","11,210","511,065","452,155","35,585","11,105","2,135","3,815","6,260"
6,Alabama's 5th,AL-05,72.8,17.4,5.1,1.7,0.6,2.4,74.8,17.0,...,"8,890","4,345","16,635","540,510","414,195","93,455","12,955","6,615","3,520","9,770"


In [8]:
demo_20 = pd.read_csv('data/demo/2019_117_acs_demo.csv')
demo_20 = demo_20.iloc[2:]
demo_20 = demo_20.set_axis(demo_cols, axis=1)
demo_20['district'] = demo_20['district'].str.replace('-AL', '-00')
demo_20.head()

,district_name,district,white_pct,black_pct,hisp_pct,asn_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,...,asn_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,asn_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama's 1st,AL-01,65.2,27.5,3.2,1.5,1.1,1.5,67.5,26.0,...,"8,140","7,600","10,570","538,420","367,555","141,810","10,985","5,950","5,820","6,300"
3,Alabama's 2nd,AL-02,61.5,31.2,3.7,1.2,0.4,1.9,64.0,30.2,...,"4,830","2,425","12,905","516,005","334,800","158,600","10,565","3,560","2,015","6,465"
4,Alabama's 3rd,AL-03,67.6,25.5,3.1,1.9,0.3,1.6,69.3,25.0,...,"7,085","1,930","11,560","540,935","381,805","137,270","9,705","4,475","1,545","6,135"
5,Alabama's 4th,AL-04,83.4,7.0,6.6,0.6,0.6,1.7,85.8,6.9,...,"2,820","4,360","11,720","511,775","452,235","35,895","11,320","2,085","3,600","6,640"
6,Alabama's 5th,AL-05,72.5,17.6,5.2,1.8,0.6,2.3,74.5,17.3,...,"9,810","4,185","16,210","546,480","417,040","95,765","13,420","7,185","3,430","9,640"


In [9]:
demo_22 = pd.read_csv('data/demo/2022_118_acs_demo.csv')
demo_22 = demo_22.iloc[2:]
demo_22 = demo_22.set_axis(demo22_cols, axis=1)
demo_22['district'] = demo_22['district'].str.replace('-AL', '-00')
demo_22.head()

,district_name,district,white_pct,black_pct,hisp_pct,asn_pct,natam_pct,pi_pct,vap_white_pct,vap_black_pct,...,asn_cit_pop,natam_cit_pop,pi_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,asn_cvap_pop,natam_cvap_pop,pi_cvap_pop
2,Alabama's 1st,AL-01,65.3,27.4,3.5,1.9,1.6,0.0,67.5,25.8,...,"11,105","11,655",105,"548,455","374,540","143,035","12,720","7,450","9,390",95
3,Alabama's 2nd,AL-02,60.6,32.0,4.1,2.1,1.0,0.0,63.2,30.5,...,"10,125","7,130",65,"540,250","347,310","168,065","11,665","6,995","5,850",50
4,Alabama's 3rd,AL-03,68.2,25.6,3.4,1.7,0.8,0.1,69.8,24.7,...,"7,730","5,665",400,"556,010","393,565","139,185","11,895","5,220","4,745",295
5,Alabama's 4th,AL-04,82.5,8.1,6.9,0.8,1.6,0.0,84.8,7.6,...,"4,480","11,380",280,"540,005","470,370","42,035","14,785","3,000","9,150",220
6,Alabama's 5th,AL-05,71.3,18.9,5.7,2.3,1.6,0.0,73.3,18.2,...,"12,765","11,135",305,"549,440","412,430","102,010","15,835","9,070","9,305",210


In [10]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} # https://gist.github.com/rogerallen/1583593

In [11]:
demo_24 = pd.read_csv('data/demo/2024_119_acs_demo_census.csv') # This is CVAP only
demo_24 = pd.pivot_table(demo_24, columns=['lntitle'], index=['geoname', 'geoid'], values='cvap_est', aggfunc='first')
demo_24 = demo_24.reset_index()
extr_tokens = demo_24['geoname'].str.extract(r"Congressional District (\(at Large\)|\d+) \(119th Congress\), ([A-za-z\s]+)")
demo_24 = pd.concat([demo_24, extr_tokens], axis=1)
demo_24 = demo_24.rename({0: 'seat_number', 1: 'state'}, axis=1)
demo_24 = demo_24[~demo_24['state'].isna()]
demo_24['district'] = demo_24['state'].astype(str).map(us_state_to_abbrev) + '-' + demo_24['seat_number'].map(lambda x: '00' if x == '(at Large)' else 
                                                                                                              (f'0{x}' if int(x) < 10 else f'{x}'))
demo_24 = demo_24.set_axis(['geoname', 'geoid', 'natam', 'natamXblack', 'natamXwhite', 'asn', 'asnXwhite', 'black', 'blackXwhite',
                           'hisp', 'pi', 'nothisp', 'multi', 'tot', 'white', 'seat_number', 'state', 'district'], axis=1)
for race in ['natam', 'asn', 'black', 'hisp', 'white', 'pi']:
    demo_24[f'cvap_{race}_pct'] = demo_24[race] / demo_24['tot'] * 100
demo_24.head(7)

,geoname,geoid,natam,natamXblack,natamXwhite,asn,asnXwhite,black,blackXwhite,hisp,...,white,seat_number,state,district,cvap_natam_pct,cvap_asn_pct,cvap_black_pct,cvap_hisp_pct,cvap_white_pct,cvap_pi_pct
0,Congressional District (at Large) (119th Congr...,5001900US0200,68109,714,23514,28657,5162,16582,3946,32606,...,347344,(at Large),Alaska,AK-00,12.640423,5.318484,3.077471,6.051383,64.463948,1.162914
1,Congressional District (at Large) (119th Congr...,5001900US1000,1133,2491,3484,22757,3153,161288,9000,52329,...,505731,(at Large),Delaware,DE-00,0.148276,2.978221,21.107845,6.848324,66.185281,0.028922
2,Congressional District (at Large) (119th Congr...,5001900US3800,21664,388,7822,4497,2076,12346,1982,19710,...,509563,(at Large),North Dakota,ND-00,3.724026,0.773031,2.122269,3.388135,87.593513,0.096264
3,Congressional District (at Large) (119th Congr...,5001900US4600,41203,389,11004,5995,2937,10007,2740,18081,...,575918,(at Large),South Dakota,SD-00,6.152199,0.895140,1.494189,2.699753,85.992821,0.036582
4,Congressional District (at Large) (119th Congr...,5001900US5000,634,153,5109,6478,2896,5264,2265,11586,...,486424,(at Large),Vermont,VT-00,0.121503,1.241479,1.008821,2.220404,93.220927,0.006133
5,Congressional District (at Large) (119th Congr...,5001900US5600,6484,55,5898,2513,2114,3452,1242,34241,...,382629,(at Large),Wyoming,WY-00,1.472659,0.570758,0.784025,7.776884,86.903464,0.065184
6,"Congressional District 1 (119th Congress), Ala...",5001900US0101,2697,426,6326,5603,1813,87067,2461,17801,...,433233,1,Alabama,AL-01,0.482094,1.001546,15.563381,3.181960,77.441168,0.036287


In [48]:
# Poll averages
pollavg = pd.read_csv('transformed/house_polling_averages.csv')
def get_poll_avg(year, state, district, party, candidate):
    # Candidate as noted in past_house_results.csv
    # Party is 'DEM' or 'REP'
    poll_df = pollavg[
        (pollavg['cycle'] == year) &
        (pollavg['state'] == state) &
        (pollavg['seat_number'] == district) &
        (pollavg['party'].isin([party, 'IND', 'OTH']))
    ]
    res_df = data[
        (data['year'] == year) &
        (data['state'] == state) &
        (data['district'] == district)
    ]

    if party == 'DEM':
        res_col = 'dem_cand'
    elif party == 'REP':
        res_col = 'rep_cand'
    else:
        raise ValueError('Not a valid party for this function')

    if res_df.shape[0] == 0:
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})
    if poll_df.shape[0] == 0:
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})

    candidate_in_res = res_df[res_col].values[0]

    if candidate_in_res[0] != '[':
        fuzzymatch = process.extractOne(candidate_in_res, poll_df['candidate_name'].values, scorer=fuzz.WRatio, score_cutoff=55)
        cand_poll_df = poll_df[poll_df['candidate_name'] == fuzzymatch]
        if cand_poll_df.shape[0] > 1:
            print(cand_poll_df['candidate_name'].values[0])
            raise ValueError(f"Something went wrong in the fuzzy match. cycle={year}, state={state}, seat={district}, party={party}.")
        avg, effn = poll_df['avg'].values[0], poll_df['effn'].values[0]
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan') if fuzzymatch is None else fuzzymatch[0],
               f'{party.lower()}_poll_avg': avg,
               f'{party.lower()}_effn': effn})
    else:
        matches = []
        averages = []
        enops = []
        if len(ast.literal_eval(candidate_in_res)) == 0:
            return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})
        for cand in ast.literal_eval(candidate_in_res):
            fuzzymatch = process.extractOne(cand, poll_df['candidate_name'].values, scorer=fuzz.WRatio, score_cutoff=55)
            cand_poll_df = poll_df[poll_df['candidate_name'] == fuzzymatch]
            if cand_poll_df.shape[0] > 1:
                print(cand_poll_df['candidate_name'].values[0])
                raise ValueError(f"Something went wrong in the fuzzy match. cycle={year}, state={state}, seat={district}, party={party}.")
            avg, effn = poll_df['avg'].values[0], poll_df['effn'].values[0]
            matches.append(float('nan') if fuzzymatch is None else fuzzymatch[0])
            averages.append(0 if fuzzymatch is None else avg)
            enops.append(0 if fuzzymatch is None else effn)
        return pd.Series({f'{party.lower()}_fuzzymatch': matches,
               f'{party.lower()}_poll_avg': np.sum(np.array(averages)),
               f'{party.lower()}_effn': max(enops)})

In [49]:
pd.set_option('display.max_columns', 100)

In [67]:
data[['dem_fuzzymatch', 'dem_poll_avg', 'dem_effn']] = data.apply(lambda x: get_poll_avg(x['year'], x['state'], x['district'],
                                                                                        'DEM', x['dem_cand']), axis=1)
data[['rep_fuzzymatch', 'rep_poll_avg', 'rep_effn']] = data.apply(lambda x: get_poll_avg(x['year'], x['state'], x['district'],
                                                                                        'REP', x['rep_cand']), axis=1)
for party in ['dem', 'rep']:
    data[f'{party}_poll_avg'].fillna(0)
    data[f'{party}_effn'].fillna(0)
data.head()

KeyboardInterrupt: 

In [ ]:
data[~data['dem_fuzzymatch'].isna()].head()

In [ ]:
data = data.drop(['dem_fuzzymatch', 'rep_fuzzymatch'], axis=1)

In [53]:
joined_dfs = []

year_to_demo_df = {
    '2018': demo_18,
    '2020': demo_20,
    '2022': demo_22,
    '2024': demo_24
}

for yr in np.unique(data['year']):
    pvi = pd.read_csv(f'transformed/pvi/past_pres_results_by{yr % 2000}dist.csv')
    prev_cyc = (yr % 2000) - ((yr % 2000) % 4)
    prev_2cyc = prev_cyc - 4
    pvi['district'] = pvi['district'].map(lambda x: x[:2] + '-00' if x[3:] == 'AL' else x)
    df = data[data['year'] == yr]
    df['seat_number'] = df['district']
    df['district'] = df['state_po'] + '-' + df['district'].map(lambda x: f'0{x}' if x < 10 else f'{x}')
    pvi = pvi[['district', f'lean_{prev_cyc}', f'lean_{prev_2cyc}']]
    df = pd.merge(left=df, right=pvi, on='district', how='left')
    df = df.rename({
        f'lean_{prev_cyc}': 'prev_lean',
        f'lean_{prev_2cyc}': 'prev2_lean'
    }, axis='columns')


    hist_gb = pd.read_csv(f'../snoutcounter-backend/averages/historical/historical_generic_ballot_{yr}.csv')
    if yr in [2018, 2020]:
        hist_appr = pd.read_csv('../snoutcounter-backend/averages/historical/historical_presidential_approval_trump_firstterm.csv')
    else:
        hist_appr = pd.read_csv('../snoutcounter-backend/averages/historical/historical_presidential_approval_biden.csv')
    gb = hist_gb.iloc[-1]['net']
    match yr:
        case 2018:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2018-11-06')]['net'].values[0]
            inc_pres = -1 # -1 for Rep, +1 for Dem
        case 2020:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2020-11-03')]['net'].values[0]
            inc_pres = -1
        case 2022:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2022-11-08')]['net'].values[0]
            inc_pres = 1
        case 2024:
            appr = hist_appr[pd.to_datetime(hist_appr['end_date']) == pd.to_datetime('2024-11-05')]['net'].values[0]
            inc_pres = 1
        case _:
            raise ValueError('Invalid year')
    df['generic_ballot_avg'] = np.full(shape=(df.shape[0],), fill_value=gb)
    df['pres_approval_avg'] = np.full(shape=(df.shape[0],), fill_value=appr)
    df['incumbent_pres'] = np.full(shape=(df.shape[0],), fill_value=inc_pres)

    ics = pd.read_csv('data/tbmics.csv')
    ics_curr = ics[(ics['Month'] == 'November') & (ics['YYYY'] == yr)]['ICS_ALL'].values[0]
    df['ics'] = np.full(shape=(df.shape[0],), fill_value=ics_curr)

    # Best to use interaction terms for presidential approval and ICS - otherwise we won't have consistent effect
    # For instance high ICS is good for Dems if Dem president is incumbent but bad for Dems if Republican president is incumbent
    df['pres_approval_avg_x_incpres'] = df['pres_approval_avg'] * df['incumbent_pres']
    df['ics_x_incpres'] = df['ics'] * df['incumbent_pres']

    # Demographics
    demo = year_to_demo_df[f'{yr}'].copy()
    demo = demo[['district', 'cvap_white_pct', 'cvap_asn_pct', 'cvap_hisp_pct', 'cvap_black_pct', 'cvap_natam_pct']]
    demo['district'] = demo['district'].astype(str)
    df['district'] = df['district'].astype(str)
    df = pd.merge(left=df, right=demo, left_on='district', right_on='district', how='left')
            
    joined_dfs.append(df)

mdata = pd.concat(joined_dfs, axis=0)
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_fuzzymatch,dem_poll_avg,dem_effn,rep_fuzzymatch,rep_poll_avg,rep_effn,seat_number,prev_lean,prev2_lean,generic_ballot_avg,pres_approval_avg,incumbent_pres,ics,pres_approval_avg_x_incpres,ics_x_incpres,cvap_white_pct,cvap_asn_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555,NaN,NaN,NaN,NaN,NaN,NaN,1,-16.192783,-14.281877,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,68.3,1.1,2.1,26.4,1.1
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938,NaN,NaN,NaN,NaN,NaN,NaN,2,-17.411483,-15.288962,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,65.3,0.6,1.9,30.5,0.4
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,False,True,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692,NaN,NaN,NaN,NaN,NaN,NaN,3,-18.001999,-14.821901,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,70.7,0.8,1.8,25.3,0.3
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,False,True,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48,529497.35,11.550741,88.449259,NaN,NaN,NaN,NaN,NaN,NaN,4,-33.277483,-27.698357,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,88.5,0.4,2.2,7.0,0.7
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,False,True,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38,1769886.91,31.266773,68.733227,NaN,NaN,NaN,NaN,NaN,NaN,5,-18.480031,-16.661366,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,76.6,1.2,2.4,17.3,0.7


In [54]:
mdata.columns.values

array(['year', 'state', 'state_po', 'special', 'district', 'dem', 'rep',
       'totalvotes', 'dem_cand', 'rep_cand', 'dem_inc', 'rep_inc',
       'dem_funds', 'rep_funds', '2party_votes', 'dem_pct_2p',
       'rep_pct_2p', 'dem_tot_funds', 'rep_tot_funds', 'tot_funds',
       'dem_funds_2p_pct', 'rep_funds_2p_pct', 'dem_fuzzymatch',
       'dem_poll_avg', 'dem_effn', 'rep_fuzzymatch', 'rep_poll_avg',
       'rep_effn', 'seat_number', 'prev_lean', 'prev2_lean',
       'generic_ballot_avg', 'pres_approval_avg', 'incumbent_pres', 'ics',
       'pres_approval_avg_x_incpres', 'ics_x_incpres', 'cvap_white_pct',
       'cvap_asn_pct', 'cvap_hisp_pct', 'cvap_black_pct',
       'cvap_natam_pct'], dtype=object)

In [55]:
mdata.shape

(1742, 42)

In [56]:
isinstance(3, int)

True

In [57]:
for party in ['dem', 'rep']:
    mdata[f'{party}_funds_2p_pct'] = mdata[f'{party}_funds_2p_pct'].fillna(0)

In [58]:
# Exclude uncontested races and races w/ all candidates from one party
mdata['no_dem_cand_flag'] = mdata['dem_cand'].map(lambda x: x == '[]')
mdata['no_rep_cand_flag'] = mdata['rep_cand'].map(lambda x: x == '[]')
uncontested = mdata[(mdata['no_dem_cand_flag']) | (mdata['no_rep_cand_flag'])]
mdata = mdata[~mdata['no_dem_cand_flag']]
mdata = mdata[~mdata['no_rep_cand_flag']]
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_fuzzymatch,dem_poll_avg,dem_effn,rep_fuzzymatch,rep_poll_avg,rep_effn,seat_number,prev_lean,prev2_lean,generic_ballot_avg,pres_approval_avg,incumbent_pres,ics,pres_approval_avg_x_incpres,ics_x_incpres,cvap_white_pct,cvap_asn_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,no_dem_cand_flag,no_rep_cand_flag
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555,NaN,NaN,NaN,NaN,NaN,NaN,1,-16.192783,-14.281877,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,68.3,1.1,2.1,26.4,1.1,False,False
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938,NaN,NaN,NaN,NaN,NaN,NaN,2,-17.411483,-15.288962,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,65.3,0.6,1.9,30.5,0.4,False,False
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,False,True,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692,NaN,NaN,NaN,NaN,NaN,NaN,3,-18.001999,-14.821901,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,70.7,0.8,1.8,25.3,0.3,False,False
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,False,True,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48,529497.35,11.550741,88.449259,NaN,NaN,NaN,NaN,NaN,NaN,4,-33.277483,-27.698357,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,88.5,0.4,2.2,7.0,0.7,False,False
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,False,True,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38,1769886.91,31.266773,68.733227,NaN,NaN,NaN,NaN,NaN,NaN,5,-18.480031,-16.661366,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,76.6,1.2,2.4,17.3,0.7,False,False


In [59]:
sum(uncontested[uncontested['year'] == 2024]['no_rep_cand_flag'])

18

In [60]:
uncontested[uncontested['year'] == 2024].shape

(38, 44)

In [61]:
for col in ['dem_inc', 'rep_inc']:
    mdata[col] = mdata[col].map(lambda x: ast.literal_eval(x))
mdata['dem_inc_any'] = mdata['dem_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))
mdata['rep_inc_any'] = mdata['rep_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))

In [62]:
mdata['dem_inc_dummy'] = mdata['dem_inc_any'].map(lambda x: 1 if x == True else 0)
mdata['rep_inc_dummy'] = mdata['rep_inc_any'].map(lambda x: 1 if x == True else 0)

In [63]:
mdata = mdata[mdata['state_po'] != 'DC']

In [64]:
mdata['pvi'] = mdata['prev_lean'] * 0.75 + mdata['prev2_lean'] * 0.25
mdata.head()

,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_fuzzymatch,dem_poll_avg,dem_effn,rep_fuzzymatch,rep_poll_avg,rep_effn,seat_number,prev_lean,prev2_lean,generic_ballot_avg,pres_approval_avg,incumbent_pres,ics,pres_approval_avg_x_incpres,ics_x_incpres,cvap_white_pct,cvap_asn_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi
0,2018,Alabama,AL,False,AL-01,89226.0,153228.0,242617,Robert Kennedy Jr,Bradley Byrne,False,True,39095.21,343761.59,242454.0,36.801208,63.198792,39095.21,343761.59,382856.80,10.211445,89.788555,NaN,NaN,NaN,NaN,NaN,NaN,1,-16.192783,-14.281877,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,68.3,1.1,2.1,26.4,1.1,False,False,False,True,0,1,-15.715056
1,2018,Alabama,AL,False,AL-02,86931.0,138879.0,226230,Tabitha Isner,Martha Roby,False,True,501490.57,799289.56,225810.0,38.497409,61.502591,501490.57,799289.56,1300780.13,38.553062,61.446938,NaN,NaN,NaN,NaN,NaN,NaN,2,-17.411483,-15.288962,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,65.3,0.6,1.9,30.5,0.4,False,False,False,True,0,1,-16.880853
2,2018,Alabama,AL,False,AL-03,83996.0,147770.0,231915,Mallory Hagan,Mike Rogers,False,True,409309.26,530372.0,231766.0,36.241727,63.758273,409309.26,530372.00,939681.26,43.558308,56.441692,NaN,NaN,NaN,NaN,NaN,NaN,3,-18.001999,-14.821901,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,70.7,0.8,1.8,25.3,0.3,False,False,False,True,0,1,-17.206974
3,2018,Alabama,AL,False,AL-04,46492.0,184255.0,230969,Lee Auman,Robert Aderholt,False,True,61160.87,468336.48,230747.0,20.148474,79.851526,61160.87,468336.48,529497.35,11.550741,88.449259,NaN,NaN,NaN,NaN,NaN,NaN,4,-33.277483,-27.698357,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,88.5,0.4,2.2,7.0,0.7,False,False,False,True,0,1,-31.882701
4,2018,Alabama,AL,False,AL-05,101388.0,159063.0,260673,Peter Joffrion,Mo Brooks,False,True,553386.53,1216500.38,260451.0,38.927860,61.072140,553386.53,1216500.38,1769886.91,31.266773,68.733227,NaN,NaN,NaN,NaN,NaN,NaN,5,-18.480031,-16.661366,-7.921785,-10.073704,-1,97.5,10.073704,-97.5,76.6,1.2,2.4,17.3,0.7,False,False,False,True,0,1,-18.025364


In [65]:
mdata.shape

(1600, 49)

In [66]:
mdata.to_csv('transformed/all_2p_house_races_trainset.csv')